In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
api_key=os.getenv("GOOGLE_API_KEY")
)

In [5]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [6]:
from toyaikit.tools import Tools

# Import your custom Gemini classes
from toyaikit_adaptation.runners import GeminiRunner
from toyaikit_adaptation.llm import GeminiClient

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

llm = GeminiClient(model="gemini-2.5-flash")

runner = GeminiRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=llm,
)

In [9]:
rec["question"]

"Is it possible to enroll in the course even if I've just found it?"

In [7]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])


=== TOOL CALL ===
name: search
arguments: {"query": "enrollment"}

=== RETRIEVED RESULTS ===



=== TOOL CALL ===
name: search
arguments: {"query": "late enrollment"}

=== RETRIEVED RESULTS ===

[1]
id: cdc3b285e5
course: llm-zoomcamp
section: General Course-Related Questions
question: Can I submit homework after the deadline, or get a deadline extension?
answer: No. We don't give individual deadline extensions, and once the homework submission form is closed you can no longer submit it — there are no late submissions. While the form is still open you can submit, even if the listed deadline has already passed.

Missing a homework won't affect your certificate: homework isn't mandatory, only passing the Capstone project is. Homework points only count toward your leaderboard rank, so you'll still appear on the leaderboard with your other submissions. 




In [14]:
result.all_messages

[Content(
   parts=[
     Part(
       text="""System instruction: You're a course teaching assistant. Answer student questions based on
 the FAQ search results. Use the search tool before answering."""
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       text="Is it possible to enroll in the course even if I've just found it?"
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'query': 'enrollment'
         },
         name='search'
       ),
       thought_signature=b"\n\xc9\x03\x01\x11M2\x0f\xae\x81\xf3\xfb\xa5\xc8\xef\x93G\x07{\x94kG~\xca\xf26\xe9\xef\xd52\x0b\x1f\x8d\xb7)h\xf4\n9\xa4\xa5\xd4\xa8r%\xc6\xe0\xf4Q\x0f@\xdd\xedA\xf9\xa9=.\xbe'\x1d\x0e\xcd\x95\x1b\x04\x80\xc18~I\xae\xf2\x8a\xe9I\\\xf0\xae\x08\xf1\xbb*\x04\x8f\x86\xb7U\x88v8R\x13\x1b\x92i\xed...'
     ),
   ],
   role='model'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         name='search',


In [15]:
import json
from google.genai import types


def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if not isinstance(message, types.Content):
            continue

        for part in message.parts or []:
            function_call = getattr(part, "function_call", None)

            if function_call is None:
                continue

            tool_calls.append(
                {
                    "name": function_call.name,
                    "arguments": json.dumps(
                        dict(function_call.args or {})
                    ),
                }
            )

    return tool_calls

In [16]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search', 'arguments': '{"query": "enrollment"}'},
 {'name': 'search', 'arguments': '{"query": "late enrollment"}'}]

In [17]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [18]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": json.dumps(tool_calls),
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': "Is it possible to enroll in the course even if I've just found it?",
 'answer_agent': "I couldn't find any information in the FAQ regarding late course enrollment. The available information discusses the flexibility of homework submissions, stating that you can submit homework while the form is still open, even if the listed deadline has passed. However, this does not directly address whether you can enroll in the course itself after an initial start date.",
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': '[{"name": "search", "arguments": "{\\"query\\": \\"enrollment\\"}"}, {"name": "search", "arguments": "{\\"query\\": \\"late enrollment\\"}"}]',
 'cost': Decimal('0.0003983'),
 'document': '74eb249bbf'}

In [19]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": json.dumps(tool_calls),
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [20]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]


=== TOOL CALL ===
name: search
arguments: {"query": "requirements for earning a course certificate"}

=== RETRIEVED RESULTS ===

[1]
id: 69d122f12e
course: llm-zoomcamp
section: General Course-Related Questions
question: Certificate: Can I follow the course in a self-paced mode and get a certificate?
answer: No, you can only get a certificate if you finish the course with a "live" cohort.

To get the certificate, you need to finish a capstone project and complete the
required peer reviews. Homework is not required. You can work through the
material and prepare your project in self-paced mode, but project submission and
peer review must happen while a live cohort is accepting them. 

[2]
id: 651ba06b34
course: llm-zoomcamp
section: General Course-Related Questions
question: Will the name I put in the certificate field be shown publicly online or shared with third parties?
answer: No. The certificate name only appears on your certificate — it isn't published online or shared with third 

In [21]:
df_agent = pd.DataFrame(agent_answers)

In [22]:
df_agent["cost"].sum()

Decimal('0.0290539')

In [24]:
df_agent.to_csv("../data/agent-answers.csv", index=False)

In [25]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [27]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [32]:
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gemini-2.5-flash"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [33]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning="The agent's answer confirms that enrollment is possible, but it misses the crucial detail from the original answer about the need to submit a project while submissions are still open to receive a certificate. The agent's answer mentions 'deadlines' but doesn't connect them to project submissions for certification.", answer_score='bad', trajectory_reasoning="The search query 'enroll in the course' is relevant to the question. It's a reasonable first and only query for this type of FAQ question.", trajectory_score='good')

In [36]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [35]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [37]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [38]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [39]:
calc_total_price(usages)

0.026846900000000003

In [40]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    39
bad     11
Name: count, dtype: int64

In [41]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    48
bad      2
Name: count, dtype: int64

In [43]:
df_agent_eval.to_csv("../data/agent-evaluations.csv", index=False)